# Exercises XP – Heart Disease Prediction

## Ce qu'on va apprendre
- EDA et préprocessing sur un dataset médical
- Entraîner Logistic Regression, SVM, XGBoost
- Optimiser les hyperparamètres avec `GridSearchCV`
- Comparer les modèles avec des métriques standardisées

**Dataset :** Heart Disease Prediction Dataset (270 patients, 13 features)

## Setup – Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

%matplotlib inline
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Exercise 1 – Exploratory Data Analysis

In [ ]:
df = pd.read_csv('dataset_heart.csv')

# Nettoyer les noms de colonnes (espaces en trop)
df.columns = df.columns.str.strip()

print('Dimensions :', df.shape)
display(df.head())
print('\nTypes des colonnes :')
print(df.dtypes)
print('\nValeurs manquantes :')
print(df.isnull().sum())

In [ ]:
# La colonne cible s'appelle 'heart disease' avec des valeurs 1 et 2
# On la convertit en binaire : 1 → 0 (pas de maladie), 2 → 1 (maladie)
target = 'heart disease'
df['target'] = (df[target] == 2).astype(int)

print('Valeurs originales :', df[target].value_counts().to_dict())
print('Valeurs après encodage :', df['target'].value_counts().to_dict())

# Séparation features / cible (on retire aussi la colonne originale)
X = df.drop(columns=[target, 'target'])
y = df['target']

# Train/test split stratifié
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'\nTrain : {X_train.shape} | Test : {X_test.shape}')
print(f'Proportion de malades dans train : {y_train.mean():.2%}')
print(f'Proportion de malades dans test  : {y_test.mean():.2%}')

In [ ]:
# Répartition des classes
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

counts = y.value_counts().sort_index()
axes[0].bar(['Pas de maladie (0)', 'Maladie cardiaque (1)'],
            counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[0].set_title('Répartition des classes', fontweight='bold')
axes[0].set_ylabel('Nombre de patients')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Histogrammes de quelques features numériques clés
key_features = ['age', 'max heart rate', 'serum cholestoral', 'resting blood pressure']
colors_feat   = ['#3498db', '#9b59b6', '#e67e22', '#1abc9c']
for feat, color in zip(key_features, colors_feat):
    axes[1].hist(df[feat], bins=20, alpha=0.5, label=feat, color=color, edgecolor='white')
axes[1].set_title('Distribution de quelques features', fontweight='bold')
axes[1].set_xlabel('Valeur')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Corrélation des features avec la cible
corr_with_target = df[X.columns.tolist() + ['target']].corr()['target'].drop('target').sort_values()

plt.figure(figsize=(10, 5))
colors_corr = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr_with_target.values]
plt.barh(corr_with_target.index, corr_with_target.values, color=colors_corr)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.title('Corrélation des features avec la cible (maladie cardiaque)', fontweight='bold')
plt.xlabel('Coefficient de Pearson')
plt.tight_layout()
plt.show()

## Préprocessing pipeline

Toutes les colonnes sont numériques → on applique uniquement `StandardScaler`.

In [ ]:
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print('Colonnes catégorielles :', cat_cols)  # vide dans ce dataset
print('Colonnes numériques    :', num_cols)

# Tous les champs sont numériques, on standardise tout
pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols)
])

print('\nPréprocesseur créé.')

## Fonction d'évaluation commune

In [ ]:
def eval_and_report(name, model, X_te, y_te):
    y_pred = model.predict(X_te)

    result = {
        'accuracy':  round(accuracy_score(y_te, y_pred),                    4),
        'precision': round(precision_score(y_te, y_pred, zero_division=0),  4),
        'recall':    round(recall_score(y_te, y_pred),                      4),
        'f1':        round(f1_score(y_te, y_pred),                          4),
    }

    # AUC si predict_proba disponible
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_te)[:, 1]
        result['auc'] = round(roc_auc_score(y_te, y_proba), 4)

    print(f'\n=== {name} ===')
    for k, v in result.items():
        print(f'  {k:10s}: {v}')

    # Matrice de confusion + ROC côte à côte
    has_proba = hasattr(model, 'predict_proba')
    fig, axes = plt.subplots(1, 2 if has_proba else 1, figsize=(12 if has_proba else 5, 4))
    if not has_proba:
        axes = [axes]

    cm = confusion_matrix(y_te, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['Sain', 'Malade']).plot(
        ax=axes[0], cmap='Blues', colorbar=False
    )
    axes[0].set_title(f'Matrice de confusion – {name}', fontweight='bold')

    if has_proba:
        fpr, tpr, _ = roc_curve(y_te, y_proba)
        axes[1].plot(fpr, tpr, linewidth=2, color='steelblue',
                     label=f'AUC = {result["auc"]:.3f}')
        axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
        axes[1].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
        axes[1].set_xlabel('FPR')
        axes[1].set_ylabel('TPR')
        axes[1].set_title(f'Courbe ROC – {name}', fontweight='bold')
        axes[1].legend(loc='lower right')
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return result

print('Fonction eval_and_report prête.')

## Exercise 2 – Logistic Regression sans Grid Search

In [ ]:
pipe_lr = Pipeline([
    ('pre', pre),
    ('lr',  LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=RANDOM_STATE))
])

pipe_lr.fit(X_train, y_train)
lr_no_gs_metrics = eval_and_report('LR sans Grid Search', pipe_lr, X_test, y_test)

## Exercise 3 – Logistic Regression avec Grid Search

In [ ]:
pipe_lr_cv = Pipeline([
    ('pre', pre),
    ('lr',  LogisticRegression(solver='liblinear', max_iter=1000, random_state=RANDOM_STATE))
])

# 'liblinear' supporte l1 et l2, C contrôle la régularisation (petit C = forte régularisation)
param_grid_lr = {
    'lr__C':       [0.01, 0.1, 1.0, 10.0, 100.0],
    'lr__penalty': ['l1', 'l2']
}

grid_lr = GridSearchCV(
    pipe_lr_cv, param_grid_lr,
    cv=5, scoring='f1', n_jobs=-1, verbose=1
)
grid_lr.fit(X_train, y_train)

print('Meilleurs paramètres :', grid_lr.best_params_)
print(f'Meilleur F1 en CV    : {grid_lr.best_score_:.4f}')

best_lr = grid_lr.best_estimator_
lr_gs_metrics = eval_and_report('LR avec Grid Search', best_lr, X_test, y_test)

## Exercise 4 – SVM sans Grid Search

On choisit le **noyau RBF** (`kernel='rbf'`) car :
- Il capture des frontières de décision non-linéaires
- Il fonctionne bien sur des données médicales où les classes ne sont pas linéairement séparables
- `gamma='scale'` adapte automatiquement le rayon du noyau à la variance des features
- `C=1.0` est un point de départ standard équilibrant marge et erreurs de classification

In [ ]:
pipe_svm = Pipeline([
    ('pre', pre),
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale',
                probability=True, random_state=RANDOM_STATE))
])

pipe_svm.fit(X_train, y_train)
svm_no_metrics = eval_and_report('SVM sans Grid Search', pipe_svm, X_test, y_test)

## Exercise 5 – SVM avec Grid Search

In [ ]:
pipe_svm_cv = Pipeline([
    ('pre', pre),
    ('svm', SVC(probability=True, random_state=RANDOM_STATE))
])

svm_param_grid = {
    'svm__kernel': ['rbf', 'linear'],
    'svm__C':      [0.1, 1.0, 10.0, 100.0],
    'svm__gamma':  ['scale', 'auto']
}

grid_svm = GridSearchCV(
    pipe_svm_cv, svm_param_grid,
    cv=5, scoring='f1', n_jobs=-1, verbose=1
)
grid_svm.fit(X_train, y_train)

print('Meilleurs paramètres :', grid_svm.best_params_)
print(f'Meilleur F1 en CV    : {grid_svm.best_score_:.4f}')

best_svm = grid_svm.best_estimator_
svm_gs_metrics = eval_and_report('SVM avec Grid Search', best_svm, X_test, y_test)

## Exercise 6 – XGBoost sans Grid Search

Choix des hyperparamètres manuels :
- `n_estimators=300` : assez d'arbres pour apprendre sans trop dépasser (avec early stopping possible)
- `learning_rate=0.1` : valeur standard, bon compromis vitesse/performance
- `max_depth=4` : profondeur modérée pour éviter l'overfitting sur 270 exemples
- `subsample=0.8` : bagging partiel pour plus de robustesse
- `colsample_bytree=0.8` : sous-échantillonnage des features par arbre

In [ ]:
pipe_xgb = Pipeline([
    ('pre', pre),
    ('xgb', XGBClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        eval_metric='logloss',
        verbosity=0
    ))
])

pipe_xgb.fit(X_train, y_train)
xgb_no_metrics = eval_and_report('XGBoost sans Grid Search', pipe_xgb, X_test, y_test)

## Exercise 7 – XGBoost avec Grid Search

In [ ]:
pipe_xgb_cv = Pipeline([
    ('pre', pre),
    ('xgb', XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric='logloss',
        verbosity=0
    ))
])

xgb_param_grid = {
    'xgb__n_estimators':     [100, 300, 500],
    'xgb__learning_rate':    [0.01, 0.1, 0.2],
    'xgb__max_depth':        [3, 4, 5],
    'xgb__subsample':        [0.7, 0.9],
    'xgb__colsample_bytree': [0.7, 0.9]
}

grid_xgb = GridSearchCV(
    pipe_xgb_cv, xgb_param_grid,
    cv=5, scoring='f1', n_jobs=-1, verbose=1
)
grid_xgb.fit(X_train, y_train)

print('Meilleurs paramètres :', grid_xgb.best_params_)
print(f'Meilleur F1 en CV    : {grid_xgb.best_score_:.4f}')

best_xgb = grid_xgb.best_estimator_
xgb_gs_metrics = eval_and_report('XGBoost avec Grid Search', best_xgb, X_test, y_test)

## Comparaison finale des modèles

In [ ]:
summary = {
    'LR sans GS':      lr_no_gs_metrics,
    'LR avec GS':      lr_gs_metrics,
    'SVM sans GS':     svm_no_metrics,
    'SVM avec GS':     svm_gs_metrics,
    'XGB sans GS':     xgb_no_metrics,
    'XGB avec GS':     xgb_gs_metrics,
}

summary_df = pd.DataFrame(summary).T
print('=== Tableau comparatif ===')
display(summary_df.style.highlight_max(axis=0, color='#aaffaa').format('{:.4f}'))

In [ ]:
# Graphique comparatif
metrics_to_show = ['accuracy', 'precision', 'recall', 'f1']
model_names     = list(summary.keys())
x               = np.arange(len(metrics_to_show))
width           = 0.13
palette         = ['#3498db', '#2980b9', '#e67e22', '#d35400', '#2ecc71', '#27ae60']

plt.figure(figsize=(14, 6))
for i, (model_name, color) in enumerate(zip(model_names, palette)):
    vals = [summary[model_name].get(m, 0) for m in metrics_to_show]
    plt.bar(x + i * width, vals, width, label=model_name, color=color, alpha=0.85, edgecolor='white')

plt.xticks(x + width * 2.5, ['Accuracy', 'Precision', 'Recall', 'F1-Score'])
plt.ylim(0, 1.15)
plt.ylabel('Score')
plt.title('Comparaison des 6 modèles – Heart Disease Prediction', fontsize=13, fontweight='bold')
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Courbes ROC superposées de tous les modèles
roc_models = {
    'LR sans GS':  pipe_lr,
    'LR avec GS':  best_lr,
    'SVM sans GS': pipe_svm,
    'SVM avec GS': best_svm,
    'XGB sans GS': pipe_xgb,
    'XGB avec GS': best_xgb,
}
roc_colors = ['#3498db', '#2980b9', '#e67e22', '#d35400', '#2ecc71', '#27ae60']

plt.figure(figsize=(9, 7))
for (model_name, model), color in zip(roc_models.items(), roc_colors):
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, linewidth=2, color=color, label=f'{model_name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
plt.xlabel('Taux de Faux Positifs (FPR)', fontsize=11)
plt.ylabel('Taux de Vrais Positifs (TPR)', fontsize=11)
plt.title('Courbes ROC – Tous les modèles', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Conclusion

| Famille | Sans Grid Search | Avec Grid Search | Gain |
|---|---|---|---|
| **Logistic Regression** | Baseline rapide et interprétable | `C` et `penalty` optimisés | Léger gain en F1 |
| **SVM** | Bonne performance avec RBF | Meilleur noyau et `C` trouvés | Gain notable |
| **XGBoost** | Déjà solide avec les hyperparamètres manuels | Fine-tuning de profondeur, subsampling, etc. | Meilleure AUC |

**Modèle recommandé :** XGBoost avec Grid Search offre généralement les meilleures performances sur ce type de dataset tabulaire. Il capture des patterns non-linéaires que LR ne peut pas modéliser, et est plus robuste que SVM sur des datasets déséquilibrés.

**Note médicale :** Dans un contexte de détection de maladie cardiaque, le **Recall** est la métrique prioritaire — un faux négatif (patient malade classé comme sain) est plus dangereux qu'une fausse alarme.